In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# NeuroCalm Voice Therapy System - Interactive Analysis\n",
    "## Explore voice analysis, stress detection, and therapeutic audio generation"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Setup and Imports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('..')\n",
    "\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import librosa\n",
    "import librosa.display\n",
    "\n",
    "from src.audio_processing import AudioProcessor\n",
    "from src.stress_detector import StressDetector, TemporalStressAnalyzer\n",
    "from src.frequency_generator import TherapeuticAudioGenerator\n",
    "from src.meditation_guide import MeditationGuide, BreathingCoach\n",
    "from src.utils import generate_synthetic_data, normalize_audio\n",
    "\n",
    "# Set style\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (15, 8)\n",
    "\n",
    "print(\"✓ Imports successful\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Generate Synthetic Voice Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate training data\n",
    "audio_samples, labels = generate_synthetic_data(num_samples=50, duration=5.0, sr=22050)\n",
    "\n",
    "print(f\"Generated {len(audio_samples)} audio samples\")\n",
    "print(f\"Labels: {set(labels)}\")\n",
    "\n",
    "# Visualize first sample\n",
    "plt.figure(figsize=(15, 4))\n",
    "plt.plot(audio_samples[0][:5000])\n",
    "plt.title(f'Sample Audio Waveform - Label: {labels[0]}')\n",
    "plt.xlabel('Sample')\n",
    "plt.ylabel('Amplitude')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Audio Feature Extraction"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize processor\n",
    "processor = AudioProcessor(sr=22050)\n",
    "\n",
    "# Extract features from first sample\n",
    "features = processor.extract_all_features(audio_samples[0])\n",
    "\n",
    "print(\"Extracted Features:\")\n",
    "for key, value in features.items():\n",
    "    if isinstance(value, np.ndarray):\n",
    "        print(f\"  {key}: shape {value.shape}\")\n",
    "    else:\n",
    "        print(f\"  {key}: {value:.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize features\n",
    "fig, axes = plt.subplots(2, 3, figsize=(18, 10))\n",
    "fig.suptitle('Extracted Audio Features', fontsize=16)\n",
    "\n",
    "# Spectral Centroid\n",
    "axes[0, 0].plot(features['spectral_centroid'])\n",
    "axes[0, 0].set_title('Spectral Centroid')\n",
    "axes[0, 0].set_xlabel('Frame')\n",
    "axes[0, 0].set_ylabel('Frequency (Hz)')\n",
    "\n",
    "# MFCC\n",
    "librosa.display.specshow(features['mfcc'], x_axis='time', ax=axes[0, 1])\n",
    "axes[0, 1].set_title('MFCC')\n",
    "axes[0, 1].set_ylabel('Coefficient')\n",
    "\n",
    "# Pitch\n",
    "axes[0, 2].plot(features['pitch'])\n",
    "axes[0, 2].set_title('Pitch Contour')\n",
    "axes[0, 2].set_xlabel('Frame')\n",
    "axes[0, 2].set_ylabel('Frequency (Hz)')\n",
    "\n",
    "# Energy\n",
    "axes[1, 0].plot(features['energy'])\n",
    "axes[1, 0].set_title('Energy (RMS)')\n",
    "axes[1, 0].set_xlabel('Frame')\n",
    "axes[1, 0].set_ylabel('Amplitude')\n",
    "\n",
    "# LPC Coefficients\n",
    "axes[1, 1].plot(features['lpc_coeffs'].T)\n",
    "axes[1, 1].set_title('LPC Coefficients')\n",
    "axes[1, 1].set_xlabel('Frame')\n",
    "axes[1, 1].set_ylabel('Coefficient Value')\n",
    "\n",
    "# Zero Crossing Rate\n",
    "axes[1, 2].plot(features['zcr'])\n",
    "axes[1, 2].set_title('Zero Crossing Rate')\n",
    "axes[1, 2].set_xlabel('Frame')\n",
    "axes[1, 2].set_ylabel('ZCR')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Train Stress Detection Model (HMM)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Extract features for all samples\n",
    "features_list = []\n",
    "for audio in audio_samples:\n",
    "    feat_vec = processor.compute_feature_vector(audio)\n",
    "    # Create sequence\n",
    "    feat_seq = np.tile(feat_vec, (10, 1))\n",
    "    features_list.append(feat_seq)\n",
    "\n",
    "print(f\"Extracted features for {len(features_list)} samples\")\n",
    "print(f\"Feature dimension: {features_list[0].shape}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train HMM\n",
    "detector = StressDetector(n_states=3)\n",
    "detector.train(features_list, labels)\n",
    "\n",
    "print(\"\\n✓ Model trained successfully\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Test Stress Detection"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test on a new sample\n",
    "test_idx = 10\n",
    "test_audio = audio_samples[test_idx]\n",
    "true_label = labels[test_idx]\n",
    "\n",
    "# Extract features\n",
    "test_features = processor.compute_feature_vector(test_audio)\n",
    "test_seq = np.tile(test_features, (10, 1))\n",
    "\n",
    "# Predict\n",
    "stress_level, state_seq, confidence = detector.predict_stress_level(test_seq)\n",
    "stress_score = detector.calculate_stress_score(test_seq)\n",
    "\n",
    "print(f\"True Label: {true_label}\")\n",
    "print(f\"Predicted: {stress_level}\")\n",
    "print(f\"Stress Score: {stress_score:.1f}/100\")\n",
    "print(f\"Confidence: {confidence*100:.1f}%\")\n",
    "print(f\"State Sequence: {state_seq}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize state probabilities\n",
    "posteriors = detector.get_state_probabilities(test_seq)\n",
    "\n",
    "plt.figure(figsize=(12, 6))\n",
    "plt.plot(posteriors[:, 0], label='Low Stress', linewidth=2)\n",
    "plt.plot(posteriors[:, 1], label='Medium Stress', linewidth=2)\n",
    "plt.plot(posteriors[:, 2], label='High Stress', linewidth=2)\n",
    "plt.xlabel('Frame')\n",
    "plt.ylabel('Probability')\n",
    "plt.title('Stress State Probabilities Over Time')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Generate Healing Frequencies"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize generator\n",
    "generator = TherapeuticAudioGenerator(sr=22050)\n",
    "\n",
    "# Generate 528 Hz healing tone\n",
    "healing_tone = generator.generate_healing_tone(\n",
    "    frequency=528,\n",
    "    duration=5.0,\n",
    "    amplitude=0.3\n",
    ")\n",
    "\n",
    "# Visualize\n",
    "plt.figure(figsize=(15, 4))\n",
    "plt.plot(healing_tone[:2000])\n",
    "plt.title('528 Hz Healing Frequency')\n",
    "plt.xlabel('Sample')\n",
    "plt.ylabel('Amplitude')\n",
    "plt.show()\n",
    "\n",
    "# Show frequency spectrum\n",
    "fft = np.fft.rfft(healing_tone)\n",
    "freqs = np.fft.rfftfreq(len(healing_tone), 1/22050)\n",
    "magnitude = np.abs(fft)\n",
    "\n",
    "plt.figure(figsize=(15, 4))\n",
    "plt.plot(freqs[:2000], magnitude[:2000])\n",
    "plt.axvline(x=528, color='r', linestyle='--', label='528 Hz')\n",
    "plt.xlabel('Frequency (Hz)')\n",
    "plt.ylabel('Magnitude')\n",
    "plt.title('Frequency Spectrum')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Create Binaural Beats"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate theta wave binaural beat\n",
    "left, right = generator.generate_binaural_beat(\n",
    "    base_freq=200,\n",
    "    beat_freq=7,  # Theta wave\n",
    "    duration=10.0\n",
    ")\n",
    "\n",
    "# Visualize stereo channels\n",
    "fig, axes = plt.subplots(2, 1, figsize=(15, 8))\n",
    "\n",
    "axes[0].plot(left[:5000])\n",
    "axes[0].set_title('Left Channel (200 Hz)')\n",
    "axes[0].set_ylabel('Amplitude')\n",
    "\n",
    "axes[1].plot(right[:5000])\n",
    "axes[1].set_title('Right Channel (207 Hz)')\n",
    "axes[1].set_xlabel('Sample')\n",
    "axes[1].set_ylabel('Amplitude')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(\"Binaural beat creates 7 Hz theta wave in the brain\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Transform Voice to Healing Frequency"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Transform test audio to 528 Hz\n",
    "transformed = generator.transform_voice_to_healing_freq(\n",
    "    test_audio,\n",
    "    target_freq=528\n",
    ")\n",
    "\n",
    "# Compare spectrograms\n",
    "fig, axes = plt.subplots(1, 2, figsize=(15, 5))\n",
    "\n",
    "D1 = librosa.amplitude_to_db(np.abs(librosa.stft(test_audio)), ref=np.max)\n",
    "librosa.display.specshow(D1, sr=22050, x_axis='time', y_axis='hz', ax=axes[0])\n",
    "axes[0].set_title('Original Voice')\n",
    "\n",
    "D2 = librosa.amplitude_to_db(np.abs(librosa.stft(transformed)), ref=np.max)\n",
    "librosa.display.specshow(D2, sr=22050, x_axis='time', y_axis='hz', ax=axes[1])\n",
    "axes[1].set_title('Transformed to 528 Hz')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Create Complete Therapeutic Session"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create stress relief session\n",
    "therapeutic = generator.create_therapeutic_session(\n",
    "    test_audio,\n",
    "    session_type='stress_relief'\n",
    ")\n",
    "\n",
    "# Visualize\n",
    "plt.figure(figsize=(15, 8))\n",
    "\n",
    "plt.subplot(2, 1, 1)\n",
    "plt.plot(therapeutic[:10000])\n",
    "plt.title('Therapeutic Audio Waveform')\n",
    "plt.ylabel('Amplitude')\n",
    "\n",
    "plt.subplot(2, 1, 2)\n",
    "D = librosa.amplitude_to_db(np.abs(librosa.stft(therapeutic)), ref=np.max)\n",
    "librosa.display.specshow(D, sr=22050, x_axis='time', y_axis='hz')\n",
    "plt.title('Therapeutic Audio Spectrogram')\n",
    "plt.colorbar(format='%+2.0f dB')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(\"Therapeutic session includes:\")\n",
    "print(\"  - Transformed voice at 528 Hz\")\n",
    "print(\"  - Alpha wave binaural beats (10 Hz)\")\n",
    "print(\"  - Healing frequency tones\")\n",
    "print(\"  - Pink noise for naturalness\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Temporal Analysis with DTW"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compare stress patterns\n",
    "sample1_features = features_list[0]\n",
    "sample2_features = features_list[1]\n",
    "\n",
    "dtw_distance = detector.compare_stress_patterns(sample1_features, sample2_features)\n",
    "\n",
    "print(f\"DTW Distance: {dtw_distance:.2f}\")\n",
    "print(f\"Labels: {labels[0]} vs {labels[1]}\")\n",
    "print(f\"Similarity: {'High' if dtw_distance < 10 else 'Low'}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Meditation Guidance"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate meditation script\n",
    "guide = MeditationGuide(sr=22050)\n",
    "\n",
    "script = guide.generate_meditation_script(\n",
    "    duration=300,  # 5 minutes\n",
    "    focus_type='breath'\n",
    ")\n",
    "\n",
    "print(\"Meditation Script:\")\n",
    "print(\"=\" * 50)\n",
    "for prompt in script:\n",
    "    print(f\"[{prompt['time']:3d}s] {prompt['text']}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 11. Summary Statistics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test model on all samples\n",
    "predictions = []\n",
    "scores = []\n",
    "\n",
    "for feat in features_list:\n",
    "    pred, _, _ = detector.predict_stress_level(feat)\n",
    "    score = detector.calculate_stress_score(feat)\n",
    "    predictions.append(pred)\n",
    "    scores.append(score)\n",
    "\n",
    "# Visualize distribution\n",
    "fig, axes = plt.subplots(1, 2, figsize=(15, 5))\n",
    "\n",
    "# Stress scores by label\n",
    "for label in set(labels):\n",
    "    label_scores = [s for s, l in zip(scores, labels) if l == label]\n",
    "    axes[0].hist(label_scores, alpha=0.5, label=label, bins=10)\n",
    "axes[0].set_xlabel('Stress Score')\n",
    "axes[0].set_ylabel('Count')\n",
    "axes[0].set_title('Distribution of Stress Scores')\n",
    "axes[0].legend()\n",
    "\n",
    "# Confusion matrix\n",
    "from sklearn.metrics import confusion_matrix\n",
    "import seaborn as sns\n",
    "\n",
    "cm = confusion_matrix(labels, predictions, labels=['low', 'medium', 'high'])\n",
    "sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],\n",
    "            xticklabels=['Low', 'Medium', 'High'],\n",
    "            yticklabels=['Low', 'Medium', 'High'])\n",
    "axes[1].set_xlabel('Predicted')\n",
    "axes[1].set_ylabel('True')\n",
    "axes[1].set_title('Confusion Matrix')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# Calculate accuracy\n",
    "accuracy = np.mean([p == l for p, l in zip(predictions, labels)])\n",
    "print(f\"\\nModel Accuracy: {accuracy*100:.1f}%\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Conclusion\n",
    "\n",
    "This notebook demonstrates:\n",
    "1. ✅ Feature extraction using FFT, LPC, MFCC\n",
    "2. ✅ HMM-based stress detection with Viterbi\n",
    "3. ✅ DTW for temporal pattern analysis\n",
    "4. ✅ Therapeutic audio generation\n",
    "5. ✅ Healing frequency transformation\n",
    "6. ✅ Binaural beats and brainwave entrainment\n",
    "7. ✅ Meditation guidance system\n",
    "\n",
    "The complete system is ready for use!"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}